# 7. Hugging Face Jobs

> **Not executed in this documentation build.** Submitting a job needs a
> Hugging Face token and is billed to an account, and the build has neither.
> The cells that would submit jobs are therefore shown as text. The few code
> cells that are executed below touch no network; they exist to check that
> every name used on this page is real. The backend itself has been run end
> to end against real Hugging Face Jobs containers; that evidence is in
> `docs/evidence/` in the repository.

With `cluster_type="huggingface"` a decorated function runs in a container
started by [Hugging Face Jobs](https://huggingface.co/docs/huggingface_hub/guides/jobs).
There is no host to log in to and no SSH: the function travels inside the job
request and the result comes back through the job's log, signed with a
per-job key.

Use it when you have no cluster, or want a GPU for an hour.

## What you need

- a Hugging Face account that may use Jobs (a paid feature at the time of
  writing), and a token with write access;
- `pip install huggingface_hub`.

## The token

Clustrix looks for a token in three places, in this order:

1. `hf_token`, given to `configure()`;
2. the environment variable `HF_TOKEN`;
3. the login stored by `hf auth login`.

Prefer 2 or 3. A token typed into a notebook is saved with the notebook.
Without a token, the first call fails before anything is sent:

In [1]:
# cluster-required: Hugging Face Jobs
import clustrix
from clustrix import cluster, configure

configure(cluster_type="huggingface")


@cluster
def hello():
    return "hello from a container"


try:
    hello()
except RuntimeError as error:
    print(error)

No HuggingFace token configured. Set hf_token in your clustrix config, export HF_TOKEN, or run `hf auth login`.


Note that the call did **not** run on this machine. `huggingface` has no
`cluster_host`, and clustrix does not fall back to local execution for it.

## Configure

```python
configure(
    cluster_type="huggingface",
    hf_namespace="your-username",      # or an organisation you may bill to
    hf_flavor="cpu-basic",
    hf_job_timeout="30m",
)

@cluster
def fit(n):
    import numpy as np
    return float(np.random.default_rng(0).normal(size=n).mean())

fit(1_000_000)
```

- `hf_namespace`: the account the job runs under and is billed to. If you
  leave it out, Hugging Face uses the account the token belongs to.
- `hf_flavor`: the hardware. The default and the timeout default are
  constants of the backend:

In [2]:
from clustrix import hf_jobs
from clustrix.modern_notebook_widget import HF_FLAVORS

print("default flavor: ", hf_jobs.DEFAULT_FLAVOR)
print("default timeout:", hf_jobs.DEFAULT_TIMEOUT)
print("flavors offered by the widget:", ", ".join(HF_FLAVORS))

default flavor:  cpu-basic
default timeout: 30m
flavors offered by the widget: cpu-basic, cpu-upgrade, cpu-performance, cpu-xl, t4-small, t4-medium, l4x1, a10g-small, a100-large, h200


  The authoritative list of flavors and prices is Hugging Face's; any flavor
  name it accepts can be given.
- `hf_job_timeout` (or `@cluster(hf_timeout=...)` for one function): how
  long the job may run, such as `"30m"` or `"2h"`. A job that reaches it is
  stopped by Hugging Face. This is the setting that bounds what a mistake can
  cost; the decorator's `time=` is not used by this backend.

Two settings can be overridden per function on the decorator: `hf_flavor`
and `hf_timeout`. The namespace and the token are set with `configure()`
only.

## GPU flavors and billing

**Jobs are billed per second of hardware time, from the moment the container
starts until it stops.** CPU flavors cost little; GPU flavors cost real money
if left running.

For that reason a GPU flavor is refused unless you have said, in the
configuration, that you intend to pay for one. Every flavor whose name does
not start with `cpu-` counts as a GPU flavor:

In [3]:
for flavor in ("cpu-basic", "cpu-xl", "t4-small", "a10g-small"):
    print(f"{flavor:<12} GPU flavor: {hf_jobs.is_gpu_flavor(flavor)}")

cpu-basic    GPU flavor: False
cpu-xl       GPU flavor: False
t4-small     GPU flavor: True
a10g-small   GPU flavor: True


```python
configure(hf_flavor="a10g-small")
fit(1_000)
# ValueError: Flavor 'a10g-small' is a GPU flavor and bills by the second. Set
# hf_allow_gpu_flavors=True to confirm you intend to pay for it; otherwise use
# a CPU flavor (default: cpu-basic).

configure(hf_flavor="a10g-small", hf_allow_gpu_flavors=True, hf_job_timeout="20m")
```

The widget's "Allow GPU hardware" checkbox is the same setting.

Before leaving a notebook, check that nothing is still running: the Jobs page
of your namespace on huggingface.co lists running jobs and can cancel them.

## Differences from SSH and SLURM

- **The environment** is a container image with the same Python minor version
  as yours, chosen automatically. Packages the function imports are installed
  when the job starts, every time; there is no environment kept between jobs.
- **A loop that can be divided becomes one job per chunk here too**, up to
  `max_parallel_jobs` (default 100), and every job is billed. Set
  `max_parallel_jobs` to a number you are willing to pay for, or use
  `@cluster(parallel=False)` to keep a function to one job.
- **No automatic input staging** (tutorial 5). The container cannot see your
  files. Pass data as arguments, or use `clustrix.data_package`: a package
  too large to travel inline is uploaded to a private dataset repository
  named `clustrix-data` that clustrix creates in your account, and stays there
  until you delete it (`package.delete()`, `clustrix.list_data_packages()`,
  `clustrix.delete_data_package(id)`).
- **Large arguments** (over 256 KB once serialized) are likewise passed
  through a private dataset repository in your account.
- **Output files are not brought back**; return the data.
- `memory`, `time` and `partition` on the decorator have no meaning here. The
  flavor fixes the hardware.

## If something looks wrong

| You see | It means |
|---|---|
| `No HuggingFace token configured` | Export `HF_TOKEN`, run `hf auth login`, or set `hf_token`. |
| `Flavor '...' is a GPU flavor and bills by the second` | Set `hf_allow_gpu_flavors=True` if you mean it. |
| `huggingface_hub is not installed` | `pip install huggingface_hub`. |
| An error from Hugging Face about payment or permissions | The namespace may not run Jobs, or the token lacks write access. |
| "unknown opcode" or a crash when the function starts | A custom `hf_image` has a different Python version from yours. Remove `hf_image`. |
| The job stops early | It reached `hf_job_timeout` (default 30 minutes). |

This is the last tutorial. The reference pages cover each setting in full.